[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/python-fundamentals-hh/blob/main/notebooks/08_landuse-data/08_01_land_cover_impervious.ipynb)


# Module 8, Lesson 1: Annual NLCD Land Cover and Impervious Surface
## Watershed summaries and change analysis with free USGS web services

### Welcome!
This module teaches you how to work with official Annual NLCD land cover and impervious surface data for watershed hydrology. You will learn the difference between a categorical land cover raster and a continuous percent-impervious raster, calculate watershed summaries correctly, and compare two years from one consistent annual product series. The notebook fetches its own raster data from a free USGS web service, so there is nothing to download or upload beyond the watershed boundary file.

### What You'll Work Through Today:
- Understand the difference between categorical and continuous rasters
- Discover the years available in Annual NLCD Collection 1
- Fetch free Annual NLCD rasters for a watershed from the official USGS service
- Validate each response as a real, aligned 30 meter raster
- Clip Annual NLCD land cover and summarize classes by area and percent
- Calculate a defensible watershed-average percent impervious value
- Use the Annual NLCD descriptor to separate roads from other built surfaces
- Compare 2001 and 2025 using one consistent annual methodology
- Export traceable summary tables and clipped rasters

### Module Structure:
1. **Mental Models** - Land cover, land use, and categorical versus continuous rasters
2. **The Data Source** - Annual NLCD and free access options
3. **Workspace Setup** - Libraries and the watershed boundary
4. **Watershed, Data Fetch, and CRS** - Year discovery, tiled WCS requests, and QA/QC
5. **Land Cover Workflow** - Clip, count, and summarize by class
6. **Percent Impervious Workflow** - Correct NoData handling and area weighting
7. **Impervious Descriptor** - Roads versus other built surfaces
8. **Change Analysis** - 2001 versus 2025
9. **Export** - Model-ready tables and clipped rasters
10. **Beyond Annual NLCD** - Coastal and local data refinements

### Prerequisites
This module assumes you have completed Module 1, Module 3, and Module 4. Module 7 is helpful background because its soil outputs pair with land cover in curve number workflows, but it is not required.


## Using AI in This Module

An AI assistant can be useful here when you ask it to explain a decision, trace data through a function, or help diagnose an output that failed a check. Keep the question tied to the code and the engineering purpose.

Try these review questions as you work:

- *"Why can I average fractional impervious percentages but not land cover class codes?"*
- *"My clipped raster has fewer valid cells than another product on the same grid. What should I inspect first?"*
- *"Show me how the NoData value changes both the numerator and denominator of a watershed average."*
- *"Trace one Annual NLCD cell from the WCS response through clipping, masking, and the final summary table."*

Read the answer critically. Confirm product definitions, units, CRS, NoData, and value ranges against the notebook and official documentation before changing code.


## Part 1: Mental Models - Land Cover, Land Use, and Two Kinds of Rasters

### Land Cover Is Not the Same as Land Use

**Land cover** is what physically covers the ground: forest, pavement, water, grass. **Land use** is how people use that land: a park and a residential yard can have the exact same grass land cover but very different land use. H&H modeling almost always needs land cover, because runoff and infiltration respond to what is physically on the ground, not to zoning. Every product in this module is a land cover product.

### A Quick Raster Recap

Module 4 introduced rasters as grids of cells, each holding a value, with a resolution, a coordinate reference system, and a NoData convention. This module builds directly on that. If any of those words feel unfamiliar, a quick look back at Module 4 will help before continuing here.

### Categorical vs. Continuous Rasters: The Idea This Module Is Built On

This is the single most important distinction in this module, so it is worth sitting with for a moment.

- A **categorical raster** stores a class code in every cell. NLCD land cover is categorical: a cell with the value 42 means "evergreen forest." The number 42 is a label, not a quantity. Averaging class codes together (adding 42 and 21 and dividing by two) produces a meaningless number. You can only count how many cells fall into each class.
- A **continuous raster** stores a measured quantity in every cell. The NLCD fractional impervious surface product is continuous: a cell with the value 37 means an estimated 37 percent of that 30 meter cell is impervious. Averaging these values together is exactly the right thing to do; that is what the product is for.

| | Categorical | Continuous |
|---|---|---|
| Example in this module | Land Cover, Impervious Descriptor | Fractional Impervious Surface |
| What a cell value means | A class code | A measured percentage |
| Correct summary | Count cells per class, convert to area | Average the values, weighted by area |
| Wrong summary | Averaging the codes | Treating it like a category |

### The Three NLCD Products Used in This Module

| Product | Type | What It Tells You |
|---|---|---|
| Land Cover | Categorical | Which land cover class each cell belongs to (forest, developed, water, and so on) |
| Fractional Impervious Surface | Continuous | The estimated percent impervious surface in each cell |
| Impervious Descriptor | Categorical | Whether an impervious cell is a road, or another kind of built surface |

The rule of thumb this module follows: use Land Cover for class summaries, use Fractional Impervious Surface whenever you need a percent impervious number, and use the Impervious Descriptor only when you specifically need to separate roads from buildings. Do not estimate percent impervious just by counting developed land cover classes; a "Developed, Open Space" cell is nowhere near 100 percent impervious, and the fractional product exists precisely so you do not have to guess.

### Why This Matters in H&H Work

Land cover and imperviousness feed directly into:

- **Curve numbers.** Module 4 built a composite curve number assuming a single hydrologic soil group. A real curve number workflow combines land cover (this module) with hydrologic soil group (Module 7) for each pixel.
- **Manning's roughness.** Land cover class is a common basis for assigning roughness zones in a hydraulic model.
- **HEC-HMS and HEC-RAS parameters.** Percent impervious is a direct input to many urban hydrology loss methods.
- **Existing versus future comparisons.** Land cover from two different years is exactly how a "what changed" analysis is built, which this module demonstrates directly.

### Key Terms

| Term | Meaning |
|---|---|
| Class code | A whole number representing one land cover category (categorical rasters only) |
| Legend | The lookup table connecting each class code to a name and a display color |
| Percent impervious | The estimated fraction of a cell's area covered by impervious surface, 0 to 100 |
| NoData | A raster's way of marking cells with no valid value; the correct NoData value differs by product, as this module will show |
| Equal-area projection | A coordinate system where every cell represents the same true ground area, which is what makes area and percentage math meaningful |
| Cell area | For a 30 meter NLCD cell: 30 x 30 = 900 square meters, which is 0.2224 acres |


## Part 2: The Data Source - Annual NLCD and Free Access

### About Annual NLCD

Annual National Land Cover Database Collection 1 is the current USGS land cover product for the conterminous United States. Collection 1.2 provides one 30 meter raster for every year from 1985 through 2025. The six-product suite includes Land Cover, Land Cover Change, Land Cover Confidence, Fractional Impervious Surface, Impervious Descriptor, and Spectral Change Day of Year.

Annual NLCD uses one consistent modeling framework across the time series. That makes comparisons such as 2001 versus 2025 more defensible than mixing separate legacy releases produced with different methods.

The official citation used in this lesson is:

> U.S. Geological Survey, 2024, Annual National Land Cover Database Collection 1 Science Products: U.S. Geological Survey data release, https://doi.org/10.5066/P94UXNTS.

### How This Notebook Gets the Data for Free

USGS distributes Annual NLCD through several systems. Full national tiles are available through EarthExplorer, ScienceBase, the MRLC download tools, and cloud storage. This notebook uses the time-enabled USGS EROS Web Coverage Service that supports the MRLC viewer.

The WCS is anonymous and free. It can return a small GeoTIFF for one year and one watershed bounding box, so students do not need an AWS account, billing information, or an API key.

The workflow follows five safeguards:

1. Discover available years from the live service.
2. Snap requests to the native 30 meter Annual NLCD grid.
3. Split large requests into manageable tiles.
4. Open every response with Rasterio and verify CRS, shape, resolution, NoData, and value range.
5. Use a local course copy first when it is available, then the live WCS, then the GitHub copy as a final fallback.

### Access Options

| Access Method | Best For |
|---|---|
| This lesson's USGS WCS workflow | Free, scripted watershed-sized requests by year |
| MRLC Viewer and Mosaic Download | Manual area-of-interest downloads |
| EarthExplorer | Official tiled archive downloads |
| ScienceBase | Citable, versioned releases |
| USGS cloud storage | Large cloud workflows that already have AWS access |
| WMS | Visualization in GIS software, not analysis of raw values |

### Today's Engineering Scenario

You are assessing development impacts in the Crow Creek area south of Cheyenne, Wyoming. A stormwater master plan needs a current watershed land cover summary, equivalent impervious area, and a consistent historical comparison. We will compare Annual NLCD 2001 with Annual NLCD 2025, the latest year in Collection 1.2.


## Part 3: Workspace Setup

The first code cell imports the tools used for tables, vectors, rasters, web requests, and plotting. Google Colab already includes these packages, so no installation step is needed.

Run the cell once. The final message confirms that every import completed. If an import fails in another environment, install only the missing package and rerun the cell.


In [ ]:
# Tabular and numerical data
import datetime
import math
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

# Spatial data
import geopandas as gpd
import rasterio
import rasterio.mask
from rasterio.io import MemoryFile
from rasterio.transform import from_origin

# Web requests
import requests

# Plotting
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

print("Libraries imported. Ready to work with Annual NLCD.")


### Upload the Watershed Boundary

The watershed ZIP is the only file students upload. It contains the shapefile components and projection information used to select the study HUC-12.

Run the cell and choose `NHD__Watershed_Boundaries_HUC_12_Selected.zip`. Check that the printed filename and file count match what you selected before continuing.


In [ ]:
# Use Colab's file picker for the one required course input
from google.colab import files

print("Please upload this file:")
print("1. NHD__Watershed_Boundaries_HUC_12_Selected.zip")
print()
print("Click 'Choose Files' below and select it.")

# The selected ZIP is placed in the current notebook session
uploaded = files.upload()

print(f"\nUploaded {len(uploaded)} file(s):")
# Echo the uploaded name so a wrong selection is caught immediately
for filename in uploaded.keys():
    print(f"   - {filename}")


## Part 4: Watershed, Data Fetch, and CRS

### Load and Check the Watershed

This lesson uses HUC-12 `101900090108`, Town of South Greeley. The next cell reads the ZIP directly, selects that watershed, and checks that the geometry has a coordinate reference system.

After running it, confirm three things:

- The watershed name is Town of South Greeley.
- The reported area is about 12,017 acres.
- A CRS is printed. A missing CRS would make every later reprojection and area calculation unreliable.


In [ ]:
# Read the watershed shapefile directly from the ZIP archive
watersheds = gpd.read_file(
    'zip://NHD__Watershed_Boundaries_HUC_12_Selected.zip'
)

# Select the HUC-12 used throughout this lesson
target_huc = '101900090108'
# Convert HUC12 to text so leading zeros and mixed column types are safe
target_watershed = watersheds[
    watersheds['HUC12'].astype(str) == target_huc
].copy()

# Stop early instead of failing later at geometry.iloc[0]
if target_watershed.empty:
    raise RuntimeError(f"HUC-12 {target_huc} was not found in the uploaded file.")
# A defined CRS is required for reprojection and area calculations
if target_watershed.crs is None:
    raise RuntimeError("The watershed file has no CRS. Confirm the ZIP includes its .prj file.")

print(f"Target watershed: {target_watershed['Name'].iloc[0]}")
print(f"Reported area: {target_watershed['AreaAcres'].iloc[0]:,.0f} acres")
print(f"Watershed CRS: {target_watershed.crs}")


### Set the Years and Request Bounds

The lesson compares 2001 with 2025. Keeping the years in named variables makes the analysis easier to update later without searching through every cell.

The watershed is reprojected to EPSG:5070 before its bounding box is calculated. A 500 meter buffer gives the raster clip enough data around the watershed edge. The printed bounds are request coordinates in meters, not latitude and longitude.

The next cell also records the native cell size, NoData value, and grid anchor used by Annual NLCD. These constants are checked again after data arrive.


In [ ]:
# Pin the analysis years so the lesson remains reproducible
BASELINE_YEAR = 2001
CURRENT_YEAR = 2025

# Native Annual NLCD grid properties from the service metadata
NLCD_CRS = "EPSG:5070"
NLCD_CELL_SIZE = 30.0
ANNUAL_NLCD_NODATA = 250
NLCD_GRID_ANCHOR = 15.0

# Reproject the watershed before requesting data in NLCD coordinates
target_bbox_crs = target_watershed.to_crs(NLCD_CRS)
minx, miny, maxx, maxy = target_bbox_crs.total_bounds

# A small buffer keeps the watershed away from the fetched raster edge
buffer_m = 500
request_bounds = (
    minx - buffer_m,
    miny - buffer_m,
    maxx + buffer_m,
    maxy + buffer_m,
)

print(f"Analysis years: {BASELINE_YEAR} and {CURRENT_YEAR}")
print(f"Watershed bounding box in {NLCD_CRS}, with a {buffer_m} m buffer:")
print(f"  X: {request_bounds[0]:.0f} to {request_bounds[2]:.0f}")
print(f"  Y: {request_bounds[1]:.0f} to {request_bounds[3]:.0f}")


### Build the Annual NLCD Retrieval Helpers

The retrieval workflow is split into five small pieces. Each piece has one job:

1. Describe the services, discover years, and align request bounds.
2. Validate a saved raster.
3. Request and inspect one WCS tile.
4. Assemble tiles and write one GeoTIFF.
5. Choose between a local file, the live USGS service, and the course fallback.

This is more code than the calculations later in the lesson because data retrieval has to handle network failures, service error documents, grid alignment, and incomplete files. You do not need to memorize it. Follow what enters each function, what the function checks, and what it returns.



#### Helper 1: Service Catalog, Year Discovery, and Grid Alignment

This first block contains configuration and two small preparation functions. `available_annual_nlcd_years()` asks the service which annual slices exist. `align_to_annual_nlcd_grid()` expands arbitrary bounds to whole native cells so separate requests share the same origin and resolution.

The grid anchor is not a buffer. It is the coordinate pattern used by the source raster. Snapping to it prevents half-cell shifts when tiles or years are compared.


In [ ]:
# Describe each time-enabled Annual NLCD service in one dictionary
#
# The workspace and layer build the WCS coverage name. The allowed
# values provide a product-specific check after a raster is opened.
ANNUAL_NLCD_SERVICES = {
    'land_cover': {
        'workspace': 'mrlc_Land-Cover-Native_conus_year_data',
        'layer': 'Land-Cover-Native_conus_year_data',
        'allowed_values': {
            11, 12, 21, 22, 23, 24, 31, 41, 42, 43,
            52, 71, 81, 82, 90, 95,
        },
    },
    'impervious': {
        'workspace': 'mrlc_Fractional-Impervious-Surface-Native_conus_year_data',
        'layer': 'Fractional-Impervious-Surface-Native_conus_year_data',
        'value_range': (0, 100),
    },
    'descriptor': {
        'workspace': 'mrlc_Impervious-Descriptor-Native_conus_year_data',
        'layer': 'Impervious-Descriptor-Native_conus_year_data',
        'allowed_values': {0, 1, 2},
    },
}

# Use a stable course copy only when local and live sources are unavailable
FALLBACK_BASE_URL = (
    "https://raw.githubusercontent.com/mohsennasab/python-fundamentals-hh/"
    "main/notebooks/08_landuse-data/data"
)

# Keep each public-service request modest. This watershed needs one tile.
WCS_TILE_CELLS = 1024
WCS_RETRIES = 3


def annual_nlcd_wcs_url(product):
    '''Return the official USGS WCS endpoint for one product.'''
    # Each product has its own GeoServer workspace
    workspace = ANNUAL_NLCD_SERVICES[product]['workspace']
    return f"https://dmsdata.cr.usgs.gov/geoserver/{workspace}/wcs"


def available_annual_nlcd_years():
    '''Read available years from the service, with an offline fallback.'''
    # Land cover and impervious products share the same annual time axis
    service = ANNUAL_NLCD_SERVICES['land_cover']
    coverage_id = f"{service['workspace']}__{service['layer']}"
    # DescribeCoverage returns XML containing one timePosition per year
    try:
        response = requests.get(
            annual_nlcd_wcs_url('land_cover'),
            params={
                'service': 'WCS',
                'version': '2.0.1',
                'request': 'DescribeCoverage',
                'coverageId': coverage_id,
            },
            timeout=30,
        )
        response.raise_for_status()
        # Extract four-digit years and remove duplicates with a set
        years = sorted({
            int(year)
            for year in re.findall(
                r'<gml:timePosition>(\d{4})', response.text
            )
        })
        # A nonempty live list is preferred because new releases appear here
        if years:
            return years
    except requests.exceptions.RequestException as error:
        print(f"  live year discovery was unavailable ({error})")
        # Timeouts and 5xx codes here come from the USGS server,
        # so reassure the student before using the fixed list
        print(
            "  (this usually means the USGS service is busy or "
            "down, not that something is wrong on your end)"
        )

    # Keep the fixed Collection 1.2 list available during a service outage
    return list(range(1985, 2026))


def align_to_annual_nlcd_grid(bounds):
    '''Snap bounds outward to native 30 meter Annual NLCD cell edges.'''
    # Unpack the requested lower-left and upper-right coordinates
    minx, miny, maxx, maxy = bounds
    anchor = NLCD_GRID_ANCHOR
    size = NLCD_CELL_SIZE
    # Floor minimums and ceil maximums so the aligned grid fully covers
    # the original request without shifting native cell boundaries
    minx = anchor + math.floor((minx - anchor) / size) * size
    miny = anchor + math.floor((miny - anchor) / size) * size
    maxx = anchor + math.ceil((maxx - anchor) / size) * size
    maxy = anchor + math.ceil((maxy - anchor) / size) * size
    # Convert the aligned distance to whole columns and rows
    width = int(round((maxx - minx) / size))
    height = int(round((maxy - miny) / size))
    # The affine transform records the upper-left corner and cell size
    transform = from_origin(minx, maxy, size, size)
    return (minx, miny, maxx, maxy), transform, width, height





The setup cell only defines data and functions. It does not contact USGS yet.

**Question to ask your AI assistant:** *"Explain why the minimum bounds use `floor`, the maximum bounds use `ceil`, and both calculations include the 15 meter grid anchor. Sketch one 30 meter cell in your explanation."*

#### Helper 2: Validate a Saved Raster

The next function treats every file as untrusted until it passes a checklist. It checks raster structure, CRS, resolution, NoData, requested coverage, and product-specific values. Collecting all problems before raising an error gives students a more useful troubleshooting message.


In [ ]:
def validate_annual_nlcd_raster(path, product, required_bounds=None):
    '''Confirm a file is a usable Annual NLCD raster for this request.'''
    # Product metadata tells us which value checks apply
    expected_crs = rasterio.crs.CRS.from_string(NLCD_CRS)
    service = ANNUAL_NLCD_SERVICES[product]

    # Opening the file is the first proof that it is a readable raster
    with rasterio.open(path) as src:
        problems = []

        # Check basic raster structure and native grid metadata
        if src.count != 1:
            problems.append(f"expected 1 band, found {src.count}")
        if src.width < 1 or src.height < 1:
            problems.append("raster has no cells")
        if src.crs != expected_crs:
            problems.append(f"expected {expected_crs}, found {src.crs}")
        if not np.allclose(src.res, (NLCD_CELL_SIZE, NLCD_CELL_SIZE), atol=0.01):
            problems.append(f"expected 30 meter cells, found {src.res}")
        if src.nodata != ANNUAL_NLCD_NODATA:
            problems.append(
                f"expected NoData={ANNUAL_NLCD_NODATA}, found {src.nodata}"
            )

        # A valid course fallback must also cover the requested watershed
        if required_bounds is not None:
            req_minx, req_miny, req_maxx, req_maxy = required_bounds
            # Allow one cell of tolerance for floating-point bounds
            tolerance = NLCD_CELL_SIZE
            if (
                src.bounds.left > req_minx + tolerance
                or src.bounds.bottom > req_miny + tolerance
                or src.bounds.right < req_maxx - tolerance
                or src.bounds.top < req_maxy - tolerance
            ):
                problems.append("raster does not cover the requested bounds")

        # Read band 1 and remove only the documented background value
        values = src.read(1)
        valid_values = values[values != ANNUAL_NLCD_NODATA]
        # Categorical products use explicit allowed codes. The continuous
        # impervious product uses a valid numeric range instead.
        if valid_values.size == 0:
            problems.append("raster contains no mapped cells")
        elif 'allowed_values' in service:
            unexpected = set(np.unique(valid_values)) - service['allowed_values']
            if unexpected:
                problems.append(f"unexpected class values {sorted(unexpected)}")
        else:
            valid_min, valid_max = service['value_range']
            if valid_values.min() < valid_min or valid_values.max() > valid_max:
                problems.append(
                    f"values fall outside {valid_min} to {valid_max}"
                )

        # Report all detected problems together so troubleshooting is faster
        if problems:
            raise ValueError("; ".join(problems))

        # Return a compact metadata summary for the status message
        return src.width, src.height, src.crs, src.res





Notice that validation depends on the product. Land cover and the descriptor use allowed class codes. Fractional imperviousness uses a numeric range from 0 to 100.

**Question to ask your AI assistant:** *"Walk through `validate_annual_nlcd_raster()` one check at a time. For each check, give one realistic failure it would catch and explain why that failure matters to a watershed summary."*

#### Helper 3: Request and Inspect One WCS Tile

One WCS request needs a product, year, projected bounding box, width, and height. The service response stays in memory until Rasterio proves it is a one-band Annual NLCD tile on the expected grid.

This check matters because an HTTP status of 200 only means the server sent a response. The response might still be an XML error document rather than a GeoTIFF.


In [ ]:
def fetch_wcs_tile(product, year, bounds, width, height):
    '''Request and validate one Annual NLCD tile in memory.'''
    service = ANNUAL_NLCD_SERVICES[product]

    # WCS 1.0.0 uses a coverage name, bounding box, output grid size,
    # coordinate system, file format, and TIME value for one annual slice
    params = {
        'service': 'WCS',
        'version': '1.0.0',
        'request': 'GetCoverage',
        'coverage': f"{service['workspace']}:{service['layer']}",
        'bbox': ','.join(str(value) for value in bounds),
        'crs': NLCD_CRS,
        'response_crs': NLCD_CRS,
        'format': 'GeoTIFF',
        'width': str(width),
        'height': str(height),
        'TIME': f'{year}-01-01',
    }

    # Keep the last failure so the final message contains useful detail
    last_error = None
    # Retry short-lived network and service failures before giving up
    for attempt in range(1, WCS_RETRIES + 1):
        try:
            # Request bytes only for this small tile and selected year
            response = requests.get(
                annual_nlcd_wcs_url(product),
                params=params,
                timeout=180,
            )
            response.raise_for_status()

            # XML service errors are not image responses, even with HTTP 200
            content_type = response.headers.get('content-type', '').lower()
            if 'tif' not in content_type and 'image' not in content_type:
                raise RuntimeError(
                    f"unexpected response type {content_type}: "
                    f"{response.text[:200]}"
                )

            # Open the response in memory before anything is written to disk
            # This is the decisive check that the response is a real raster
            with MemoryFile(response.content) as memory_file:
                with memory_file.open() as src:
                    if src.count != 1:
                        raise RuntimeError(f"expected 1 band, found {src.count}")
                    if src.crs is None or src.crs.to_epsg() != 5070:
                        raise RuntimeError(f"unexpected CRS {src.crs}")
                    if src.shape != (height, width):
                        raise RuntimeError(
                            f"expected {(height, width)}, found {src.shape}"
                        )
                    if src.nodata != ANNUAL_NLCD_NODATA:
                        raise RuntimeError(
                            f"expected NoData=250, found {src.nodata}"
                        )
                    data = src.read(1)
                    tile_transform = src.transform
                    try:
                        colormap = src.colormap(1)
                    except ValueError:
                        colormap = None
            # A successful tile needs no further retries
            return data, tile_transform, colormap
        except (
            requests.exceptions.RequestException,
            rasterio.errors.RasterioIOError,
            RuntimeError,
        ) as error:
            last_error = error
            print(f"    tile attempt {attempt} failed ({error})")

    raise RuntimeError(
        f"Could not download the {product} tile after {WCS_RETRIES} attempts. "
        f"Details: {last_error}"
    )





The tile function returns three items: the cell values, the tile transform, and an optional color table. It retries only failures that occur while requesting or opening the tile.

**Question to ask your AI assistant:** *"Trace the WCS parameters in `fetch_wcs_tile()`. Explain what `bbox`, `width`, `height`, `crs`, and `TIME` control, and what could go wrong if one does not match the others."*

#### Helper 4: Build a Mosaic from Validated Tiles

Small watersheds need one tile. Larger bounding boxes are divided into blocks so each public-service request remains manageable. Every tile is placed into a NoData-filled array using the transform returned by USGS.

Placing tiles by georeferencing is safer than assuming the service returned exactly the coordinates implied by loop position.


In [ ]:
def download_annual_nlcd_raster(filename, product, year, bounds):
    '''Download aligned tiles, build one mosaic, and save a GeoTIFF.'''
    # Build the exact output grid before making any requests
    aligned_bounds, transform, width, height = align_to_annual_nlcd_grid(bounds)
    minx, miny, maxx, maxy = aligned_bounds
    # Start with a NoData-filled array. Valid tiles replace these cells.
    mosaic = np.full(
        (height, width), ANNUAL_NLCD_NODATA, dtype=np.uint8
    )
    colormap = None

    # Describe each tile by its output offsets, dimensions, and bounds
    tiles = []
    # Walk across the output grid in blocks no larger than WCS_TILE_CELLS
    for row_offset in range(0, height, WCS_TILE_CELLS):
        for col_offset in range(0, width, WCS_TILE_CELLS):
            tile_height = min(WCS_TILE_CELLS, height - row_offset)
            tile_width = min(WCS_TILE_CELLS, width - col_offset)
            # Convert array offsets back to projected map coordinates
            tile_minx = minx + col_offset * NLCD_CELL_SIZE
            tile_maxx = tile_minx + tile_width * NLCD_CELL_SIZE
            tile_maxy = maxy - row_offset * NLCD_CELL_SIZE
            tile_miny = tile_maxy - tile_height * NLCD_CELL_SIZE
            tiles.append((
                row_offset, col_offset, tile_width, tile_height,
                (tile_minx, tile_miny, tile_maxx, tile_maxy),
            ))

    print(
        f"  requesting {width} x {height} cells in "
        f"{len(tiles)} tile(s) from USGS"
    )
    # Each tile is its own request, so a bigger area means a
    # noticeably longer wait -- say so up front
    if len(tiles) > 1:
        print(
            "  note: larger areas need more tiles, so this "
            "download will take longer"
        )
    # Download one tile at a time so progress and failures stay visible
    for tile_number, tile in enumerate(tiles, start=1):
        row_offset, col_offset, tile_width, tile_height, tile_bounds = tile
        data, tile_transform, tile_colormap = fetch_wcs_tile(
            product, year, tile_bounds, tile_width, tile_height
        )

        # Derive placement from the returned transform. This protects the
        # mosaic from a one-cell service shift or rounding difference.
        col_start = int(round(
            (tile_transform.c - transform.c) / NLCD_CELL_SIZE
        ))
        row_start = int(round(
            (transform.f - tile_transform.f) / NLCD_CELL_SIZE
        ))
        mosaic[
            row_start:row_start + data.shape[0],
            col_start:col_start + data.shape[1],
        ] = data
        # One source color table is enough for the completed raster
        if colormap is None and tile_colormap is not None:
            colormap = tile_colormap
        print(f"    validated tile {tile_number} of {len(tiles)}")

    # Write one compressed, georeferenced GeoTIFF after every tile passes
    with rasterio.open(
        filename,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=rasterio.uint8,
        crs=NLCD_CRS,
        transform=transform,
        nodata=ANNUAL_NLCD_NODATA,
        compress='lzw',
    ) as dst:
        # The mosaic is a two-dimensional array written to raster band 1
        dst.write(mosaic, 1)
        if colormap is not None:
            dst.write_colormap(1, colormap)





The mosaic is written only after all tiles have opened and passed their checks. The output GeoTIFF keeps the native CRS, transform, 30 meter cells, NoData value, compression, and color table.

**Question to ask your AI assistant:** *"Explain how `tile_transform.c` and `tile_transform.f` determine the destination column and row. Why is this safer than using only `row_offset` and `col_offset`?"*

#### Helper 5: Choose a Data Source

The final wrapper controls source order:

1. Use an existing local file when it covers the request and passes validation.
2. Request the official USGS WCS.
3. Use the course GitHub copy if the live service is unavailable.

Every source passes through the same validator. A fallback is not accepted merely because it exists.


In [ ]:
def fetch_annual_nlcd_raster(filename, product, year, bounds):
    '''Prepare one raster using local, live WCS, then GitHub sources.'''
    # Normalize the output path and the bounds used for coverage checks
    output_path = Path(filename)
    aligned_bounds, _, _, _ = align_to_annual_nlcd_grid(bounds)

    # Source 1: use an existing validated file to avoid network traffic
    local_candidates = [
        output_path,
        Path('data') / filename,
        Path('notebooks/08_landuse-data/data') / filename,
    ]
    # Check candidates in order and stop at the first usable file
    for candidate in local_candidates:
        if not candidate.exists():
            continue
        try:
            # A file that opens is not enough. It must pass all metadata,
            # value, and requested-coverage checks.
            validate_annual_nlcd_raster(
                candidate, product, required_bounds=aligned_bounds
            )
            # Copy course data to the notebook working directory when needed
            if candidate.resolve() != output_path.resolve():
                shutil.copyfile(candidate, output_path)
            print(f"  using validated local file: {candidate}")
            return
        except (
            OSError,
            rasterio.errors.RasterioIOError,
            ValueError,
        ) as error:
            print(f"  local candidate {candidate} was not usable ({error})")

    # Source 2: request the official service when local files are unsuitable
    try:
        download_annual_nlcd_raster(
            filename, product, year, bounds
        )
        # Validate the completed mosaic before returning it to the lesson
        width, height, crs, resolution = validate_annual_nlcd_raster(
            output_path, product, required_bounds=aligned_bounds
        )
        print(
            f"  fetched and validated from USGS WCS "
            f"({width} x {height}, {crs}, {resolution})"
        )
        return
    except (
        requests.exceptions.RequestException,
        rasterio.errors.RasterioIOError,
        RuntimeError,
        ValueError,
    ) as error:
        print(f"  live WCS request failed ({error})")
        # 500/503 responses and timeouts are produced by the USGS
        # server, so make clear the notebook is not the problem
        print(
            "  (500/503 errors and timeouts come from the USGS "
            "server, not from this notebook or your inputs)"
        )

    # Source 3: use the GitHub course copy only after the live request fails
    fallback_url = f"{FALLBACK_BASE_URL}/{filename}"
    print("  trying the course repository fallback...")
    try:
        response = requests.get(fallback_url, timeout=60)
        response.raise_for_status()
        # Save the response, then subject it to the same validation checks
        output_path.write_bytes(response.content)
        width, height, crs, resolution = validate_annual_nlcd_raster(
            output_path, product, required_bounds=aligned_bounds
        )
        print(
            f"  fetched and validated from GitHub "
            f"({width} x {height}, {crs}, {resolution})"
        )
    except (
        OSError,
        requests.exceptions.RequestException,
        rasterio.errors.RasterioIOError,
        ValueError,
    ) as error:
        raise RuntimeError(
            f"No usable source was available for {filename}. Check the "
            "internet connection or confirm the local course file is present."
        ) from error


print("Annual NLCD fetch helpers are ready.")



The retrieval helpers are now defined, but no raster has been requested. The next cell calls the wrapper five times and prints which source was selected for each product.

**Question to ask your AI assistant:** *"Trace what happens when a local file exists but does not cover the requested watershed, then the WCS returns an XML error, and the GitHub fallback is valid. List the validation and exception paths in order."*


### Run the Five Raster Requests

The next cell checks the available time axis, confirms both selected years, and prepares five files:

- Land cover for the baseline and current years
- Fractional imperviousness for the baseline and current years
- Impervious descriptor for the current year

Read each status line. `using validated local file` means the local-first path worked. `fetched and validated from USGS WCS` means the live path worked. Either source must pass the same checks.

Two things to keep in mind while this cell runs. First, the USGS service has busy periods. If you see `500` or `503` errors or timeouts in the output, that's the server struggling, not a mistake in your notebook -- the helpers retry each tile and then fall back to the course copy on GitHub, so the lesson keeps moving. Second, download time grows with the request area. A bigger watershed (or a larger `buffer_m`) means more 30 m cells and more tiles, and each tile is a separate request. During a service outage the retries add further delay, so give a large area several minutes before assuming the cell is stuck.


In [ ]:
# Discover the service time axis and confirm both selected years exist
available_years = available_annual_nlcd_years()
print(
    f"Annual NLCD years available: {available_years[0]} "
    f"through {available_years[-1]}"
)

# Fail clearly if either pinned lesson year is missing
for selected_year in [BASELINE_YEAR, CURRENT_YEAR]:
    if selected_year not in available_years:
        raise RuntimeError(
            f"Annual NLCD year {selected_year} is not available."
        )

# filename -> (product name used by the helper, year)
# Pair each output filename with the service product and requested year
nlcd_products = {
    f'annual_nlcd_{BASELINE_YEAR}_land_cover.tif':
        ('land_cover', BASELINE_YEAR),
    f'annual_nlcd_{CURRENT_YEAR}_land_cover.tif':
        ('land_cover', CURRENT_YEAR),
    f'annual_nlcd_{BASELINE_YEAR}_impervious.tif':
        ('impervious', BASELINE_YEAR),
    f'annual_nlcd_{CURRENT_YEAR}_impervious.tif':
        ('impervious', CURRENT_YEAR),
    f'annual_nlcd_{CURRENT_YEAR}_impervious_descriptor.tif':
        ('descriptor', CURRENT_YEAR),
}

print("\nPreparing Annual NLCD rasters for this watershed's extent...")
# The wrapper prints whether each file came from local data, USGS, or GitHub
for filename, (product, year) in nlcd_products.items():
    print(f"\n{filename}:")
    fetch_annual_nlcd_raster(
        filename, product, year, request_bounds
    )

print("\nAll five Annual NLCD rasters are ready.")


### Inspect the Current Land Cover Raster

Retrieval is complete only when every product reports a validated source. Now inspect one raster before analysis.

The next cell reads CRS, resolution, transform, and NoData. Cell area comes from the affine transform rather than a hard-coded 30 by 30 assumption. The expected result is EPSG:5070, approximately 30 meter cells, 900 square meters per cell, and NoData 250.


In [ ]:
# Inspect the current land cover raster before using it
lc_current_path = f'annual_nlcd_{CURRENT_YEAR}_land_cover.tif'

# Read metadata first. Do not begin calculations before checking the file.
with rasterio.open(lc_current_path) as src:
    lc_crs = src.crs
    lc_res = src.res
    lc_nodata = src.nodata
    lc_source_transform = src.transform

# The affine determinant uses all transform terms, including rotation
# terms if another raster ever contains them
cell_area_m2 = abs(
    lc_source_transform.a * lc_source_transform.e
    - lc_source_transform.b * lc_source_transform.d
)
cell_area_acres = cell_area_m2 / 4046.86

print(f"Land cover raster CRS: {lc_crs}")
print(f"Resolution: {lc_res[0]:.2f} x {lc_res[1]:.2f} meters")
print(f"Cell area from raster metadata: {cell_area_m2:,.0f} square meters")
print(f"Declared NoData value: {lc_nodata}")


### Why NLCD Uses an Equal-Area Projection

The land cover raster is in a Conterminous US Albers Equal-Area projection, not the geographic latitude/longitude system. This is deliberate. In an equal-area projection, every cell covers the same true ground area (900 square meters for a 30 meter NLCD cell), everywhere in the country. That is exactly what area and percentage calculations need. A geographic CRS would distort cell area depending on latitude, which would quietly bias every area calculation in this module.

### Reproject the Vector, Not the Raster

The watershed boundary is in a different CRS than the land cover raster. There are two ways to fix this, and only one of them is a good idea:

- **Reproject the vector watershed boundary to match the raster's CRS.** This is exact. A polygon boundary is a mathematical shape, and reprojecting it just recalculates its coordinates.
- **Reproject the raster to match the vector's CRS.** This resamples every pixel, which for a categorical raster like land cover means class codes get blended or reassigned at cell edges. For a percent-impervious raster it introduces a similar distortion. Either way, you would be changing the data to avoid changing the boundary, which is backwards.

This module always reprojects vectors to match the raster CRS, never the other way around.

 **Try asking your AI assistant:** *"Why is it better to reproject a watershed boundary to match a raster's CRS, instead of reprojecting the raster to match the vector? What actually happens to the data in each case?"*

After the reprojection cell, compare the printed watershed area with the source attribute. Small rounding differences are normal. A large difference suggests a CRS or geometry problem.


In [ ]:
# Reproject the watershed boundary to match the land cover raster's CRS
target_reprojected = target_watershed.to_crs(lc_crs)

print(f"Watershed CRS is now: {target_reprojected.crs}")

# Recompute area from the reprojected polygon, in acres and square miles
# EPSG:5070 uses meters, so area comes back in square meters
area_m2 = target_reprojected.geometry.iloc[0].area
area_acres = area_m2 / 4046.86
area_sqmi = area_acres / 640

print(f"Watershed area: {area_acres:,.0f} acres ({area_sqmi:.1f} square miles)")


## Part 5: Current Land Cover - From Pixels to a Class Table

### Step 1: Clip the Raster

Clipping keeps source cell values but replaces cells outside the watershed with background 250. The watershed geometry must already be in EPSG:5070.

The printed array shape includes the rectangular crop around the watershed, so `Total cells` is larger than the valid watershed cell count calculated next.


In [ ]:
# Rasterio needs the watershed geometry in the raster's CRS
# Rasterio expects one or more GeoJSON-like geometry mappings in a list
watershed_geom = [
    target_reprojected.geometry.iloc[0].__geo_interface__
]

# Clip the current Annual NLCD land cover raster
with rasterio.open(lc_current_path) as src:
    lc_clipped, lc_transform = rasterio.mask.mask(
        src,
        watershed_geom,
        crop=True,
        nodata=ANNUAL_NLCD_NODATA,
    )

# The returned array is shaped as band, row, column. Use its first band.
lc_array = lc_clipped[0]
print(f"Clipped array shape: {lc_array.shape}")
print(f"Total cells in clipped array: {lc_array.size}")


### Step 2: Count Cells by Class

Land cover is categorical, so count class codes instead of averaging them. `numpy.unique(..., return_counts=True)` returns each code present and its cell count.

Exclude only Annual NLCD background 250. Class code 0 is not used by Annual NLCD land cover, but 0 is valid in the impervious products later in the lesson.

After running the cell, scan the codes for unexpected values before applying the legend.


In [ ]:
# Count each categorical class, excluding only the documented background
# np.unique returns each code once and counts how many cells contain it
class_values, pixel_counts = np.unique(
    lc_array[lc_array != ANNUAL_NLCD_NODATA],
    return_counts=True,
)

print(f"Found {len(class_values)} land cover classes in the watershed")
for code_value, count in zip(class_values, pixel_counts):
    print(f"  code {code_value}: {count} pixels")


### Step 3: Attach the Annual NLCD Legend

The previous output contains numeric labels. The next cell defines the official class names and colors used to turn those labels into an understandable table and map.

The dictionary includes all 16 CONUS classes, even though this watershed contains fewer. Keeping a complete legend lets the same code work for another watershed.


In [ ]:
# Official Annual NLCD land cover class names
nlcd_classes = {
    11: 'Open Water',
    12: 'Perennial Ice/Snow',
    21: 'Developed, Open Space',
    22: 'Developed, Low Intensity',
    23: 'Developed, Medium Intensity',
    24: 'Developed, High Intensity',
    31: 'Barren Land',
    41: 'Deciduous Forest',
    42: 'Evergreen Forest',
    43: 'Mixed Forest',
    52: 'Shrub/Scrub',
    71: 'Grassland/Herbaceous',
    81: 'Pasture/Hay',
    82: 'Cultivated Crops',
    90: 'Woody Wetlands',
    95: 'Emergent Herbaceous Wetlands',
}

# Official display colors, one per class code
nlcd_colors = {
    11: '#466b9f', 12: '#d1def8', 21: '#dec5c5',
    22: '#d99282', 23: '#eb0000', 24: '#ab0000',
    31: '#b3ac9f', 41: '#68ab5f', 42: '#1c5f2c',
    43: '#b5c58f', 52: '#ccb879', 71: '#dfdfc2',
    81: '#dcd939', 82: '#ab6c28', 90: '#b8d9eb',
    95: '#6c9fb8',
}

print(f"Legend covers {len(nlcd_classes)} classes")


### Step 4: Convert Counts to Area and Percent

Every valid cell has the same area in EPSG:5070. Multiply cell count by cell area for acres, then divide each count by total valid cells for percent watershed.

Review the table after running the cell:

- Percentages should total about 100 percent.
- Areas should sum to about the watershed area.
- Dominant classes should make sense for the mapped setting.


In [ ]:

# The denominator is the number of valid classified cells
total_pixels = pixel_counts.sum()

# Build one row per land cover code found in the watershed
lc_summary = pd.DataFrame({
    'nlcd_code': class_values,
    'nlcd_class': [
        nlcd_classes.get(int(code), f'Unknown ({code})')
        for code in class_values
    ],
    'pixel_count': pixel_counts,
})

# Equal-area cells convert counts directly to acres
lc_summary['area_acres'] = (
    lc_summary['pixel_count'] * cell_area_acres
)

# Class percent uses valid classified cells as its denominator
lc_summary['percent_watershed'] = (
    100 * lc_summary['pixel_count'] / total_pixels
)

# Place the dominant classes first for easier review
lc_summary = lc_summary.sort_values(
    'percent_watershed', ascending=False
).reset_index(drop=True)

print(lc_summary[[
    'nlcd_class', 'area_acres', 'percent_watershed'
]].round(1))


### Step 5: Check Raster Area Against Polygon Area

This check compares two independent area estimates. The vector polygon gives geometric area. The raster gives valid cell count times cell area.

Small differences are expected because square cell centers approximate an irregular boundary. The cell prints a warning when the difference exceeds 2 percent.


In [ ]:

# Sum classified cell area and compare it with vector polygon area
raster_total_acres = lc_summary['area_acres'].sum()
area_difference_acres = abs(area_acres - raster_total_acres)
area_difference_percent = (
    100 * area_difference_acres / area_acres
)

print(f"Watershed area from polygon geometry: {area_acres:,.0f} acres")
print(
    f"Watershed area from classified raster pixels: "
    f"{raster_total_acres:,.0f} acres"
)
print(
    f"Difference: {area_difference_acres:,.0f} acres "
    f"({area_difference_percent:.1f}% of watershed area)"
)

if area_difference_percent > 2:
    print(
        "WARNING: Raster and polygon area differ by more than 2%. "
        "Check CRS, NoData, and clipping."
    )
else:
    print("Raster-to-polygon area QA/QC passed.")


If the area check passes, the class percentages have a defensible denominator. If it fails, stop before mapping or exporting results and inspect CRS, NoData, and the clip geometry.

### Step 6: Map the Clipped Land Cover

The plotting code converts sparse NLCD class codes into consecutive display positions while preserving official colors. This conversion is for display only. The original raster values remain unchanged.


In [ ]:
# Build a categorical colormap for the classes present here
present_codes = sorted(class_values.tolist())
colors_in_order = [nlcd_colors[code] for code in present_codes]
cmap = ListedColormap(colors_in_order)

# Convert raw class codes to consecutive plotting positions
code_to_index = {
    code: index for index, code in enumerate(present_codes)
}
lc_display = np.vectorize(
    lambda value: code_to_index.get(value, -1)
)(lc_array)
lc_display = np.ma.masked_where(
    lc_array == ANNUAL_NLCD_NODATA, lc_display
)

fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(
    lc_display,
    cmap=cmap,
    vmin=0,
    vmax=len(present_codes) - 1,
)

# Build one legend patch for every class visible on this map
legend_patches = [
    Patch(facecolor=nlcd_colors[code], label=nlcd_classes[code])
    for code in present_codes
]
ax.legend(
    handles=legend_patches,
    loc='center left',
    bbox_to_anchor=(1.0, 0.5),
    fontsize=9,
    frameon=True,
)

# Use the selected year and watershed name in the map title
ax.set_title(
    f"Annual NLCD {CURRENT_YEAR} Land Cover\n"
    f"{target_watershed['Name'].iloc[0]}"
)
ax.set_xticks([])
ax.set_yticks([])

# Tight layout leaves room for the legend outside the map axes
plt.tight_layout()
plt.show()


### What the Map Shows

Grassland dominates the watershed, while developed classes form a substantial pattern around Town of South Greeley. The map supports spatial review, while the table provides the defensible area totals.

Look for isolated colors, unexpected classes, or abrupt blank areas. Those can reveal a legend omission, NoData mistake, or clipping problem that may not be obvious from the summary table.

**Question to ask your AI assistant:** *"Compare the land cover map with `lc_summary`. Explain why the map is useful for spatial QA/QC even though the table contains the exact class areas."*


## Part 6: Fractional Impervious Surface - A NoData Mistake Worth Knowing

### First, the Wrong Way

The fractional impervious raster is continuous, so the correct watershed summary is an area-weighted average. A tempting first attempt is to treat 0 as missing because 0 often means background in categorical rasters. Run that incorrect approach once so you can see why product documentation and valid-area checks matter.


In [ ]:
# Open the current fractional impervious product
imp_current_path = f'annual_nlcd_{CURRENT_YEAR}_impervious.tif'

# This demonstration deliberately assigns 0 as the clip background
with rasterio.open(imp_current_path) as src:
    imp_clipped_wrong, _ = rasterio.mask.mask(
        src, watershed_geom, crop=True, nodata=0
    )

# Select band 1, then repeat the incorrect assumption by removing zeros
imp_array_wrong = imp_clipped_wrong[0]

# This is intentionally wrong: 0 means 0 percent impervious, not NoData
valid_wrong = imp_array_wrong[imp_array_wrong != 0]
print(f"Valid cell count after incorrectly excluding 0: {valid_wrong.size}")
print(f"Incorrect mean percent impervious: {valid_wrong.mean():.2f}%")


### Investigate the Wrong Result

The first calculation excludes every zero-percent cell. Compare its valid count with the land cover valid count from Part 5. The two products share the same grid and watershed, so a large difference is a warning.

Annual NLCD Collection 1.2 defines 250 as background. Fractional impervious values from 0 through 100 are valid. The next cell repeats the clip using 250 and checks that land cover and impervious grids align.


In [ ]:
# Repeat the clip with the documented Annual NLCD background value
with rasterio.open(imp_current_path) as src:
    imp_clipped, imp_transform = rasterio.mask.mask(
        src,
        watershed_geom,
        crop=True,
        nodata=ANNUAL_NLCD_NODATA,
    )
    imp_source_transform = src.transform

imp_array = imp_clipped[0]
# Keep every value from 0 through 100 and exclude only background 250
valid_imp_mask = imp_array != ANNUAL_NLCD_NODATA

print(
    f"Background cells inside cropped array: "
    f"{(imp_array == ANNUAL_NLCD_NODATA).sum()}"
)
print(f"Valid impervious cell count: {valid_imp_mask.sum()}")
print(
    f"Valid value range: {imp_array[valid_imp_mask].min()} "
    f"to {imp_array[valid_imp_mask].max()} percent"
)

# Shape and transform must match before cells are compared one for one
if (
    imp_array.shape != lc_array.shape
    or not imp_transform.almost_equals(lc_transform)
):
    raise RuntimeError(
        "Land cover and impervious clips are not on the same grid."
    )
print("Land cover and impervious grid alignment QA/QC passed.")


The corrected output should show values from 0 through 100 and a grid-alignment pass. The valid impervious cell count should match the valid land cover count.

### Compare the Wrong and Correct Means

The next cell uses the two masks on the same raster. This isolates the effect of the NoData decision from every other part of the workflow.


In [ ]:

# Apply the corrected mask and convert values to floating point for averaging
imp_valid = imp_array[valid_imp_mask].astype(float)

# Compare the two methods using the same watershed and raster
mean_impervious_wrong = valid_wrong.mean()
mean_impervious_correct = imp_valid.mean()
overstatement_points = (
    mean_impervious_wrong - mean_impervious_correct
)

print(
    f"Wrong approach, masked on 0:      "
    f"{mean_impervious_wrong:.2f}% mean impervious"
)
print(
    f"Correct approach, masked on 250: "
    f"{mean_impervious_correct:.2f}% mean impervious"
)
print(
    f"\nThe wrong approach overstated watershed imperviousness by "
    f"{overstatement_points:.1f} percentage points."
)


### Why the Mean Changed So Much

Removing zero-percent cells barely changes the impervious-area numerator, but it greatly reduces the valid-area denominator. The resulting mean represents only cells with some imperviousness, not the whole watershed.

### Calculate Equivalent Impervious Area

The next cell writes the area weighting explicitly:

1. Convert each valid percentage to a fraction.
2. Multiply by cell area.
3. Sum equivalent impervious area.
4. Divide by total valid mapped area.

Equal-area cells make this result equal to the simple mean, but the explicit form also produces equivalent impervious acres and makes the denominator visible.


In [ ]:
# Use the impervious raster's own affine transform for its cell area
impervious_cell_area_m2 = abs(
    imp_source_transform.a * imp_source_transform.e
    - imp_source_transform.b * imp_source_transform.d
)

# Each cell contributes its fractional imperviousness times cell area
impervious_area_per_cell_m2 = (
    imp_valid / 100
) * impervious_cell_area_m2

# Sum fractional contributions to get equivalent impervious area
total_impervious_area_m2 = impervious_area_per_cell_m2.sum()
total_impervious_area_acres = total_impervious_area_m2 / 4046.86

# The denominator includes all valid cells, including 0 percent cells
total_valid_area_m2 = imp_valid.size * impervious_cell_area_m2
total_valid_area_acres = total_valid_area_m2 / 4046.86

# Divide equivalent impervious area by total valid mapped area
percent_impervious_current = (
    100 * total_impervious_area_m2 / total_valid_area_m2
)

print(f"Total valid area: {total_valid_area_acres:,.0f} acres")
print(
    f"Equivalent impervious area: "
    f"{total_impervious_area_acres:,.0f} acres"
)
print(
    f"Watershed-average percent impervious: "
    f"{percent_impervious_current:.2f}%"
)


**Question to ask your AI assistant:** *"Use the printed valid area and equivalent impervious area to reproduce the watershed-average percentage by hand. Then explain why this value is not necessarily directly connected impervious area."*

### Engineering Caution: Check NoData, Do Not Assume It

Annual NLCD uses 250 as background for the three products in this lesson. Zero is valid in the fractional impervious and descriptor rasters. Keep the product version, user guide, and file metadata together when deciding what to mask.


## Part 7: Impervious Descriptor - Roads or Other Built Surfaces?

The descriptor is categorical: 0 is non-urban or not impervious, 1 is roads, and 2 is urban or other built surfaces. Background is 250.

Pixel counts describe land coverage, but runoff interpretation needs the impervious contribution. The next cell pairs each descriptor code with the fractional impervious percentage in the same cell. It then reports both mapped cell percent and equivalent impervious-area percent.

Before combining the arrays, the code requires matching shape and transform.


In [ ]:
# Open the current descriptor product for the same watershed and year
desc_path = f'annual_nlcd_{CURRENT_YEAR}_impervious_descriptor.tif'

with rasterio.open(desc_path) as src:
    desc_clipped, desc_transform = rasterio.mask.mask(
        src,
        watershed_geom,
        crop=True,
        nodata=ANNUAL_NLCD_NODATA,
    )
    descriptor_source_transform = src.transform

# Select band 1 after clipping with the documented background value
desc_array = desc_clipped[0]

# Descriptor codes and impervious percentages are paired cell by cell
# only after shape and transform confirm the grids are identical
if (
    desc_array.shape != imp_array.shape
    or not desc_transform.almost_equals(imp_transform)
):
    raise RuntimeError(
        "Descriptor and impervious rasters are not on the same grid."
    )

# Read descriptor cell area from its own affine transform
descriptor_cell_area_m2 = abs(
    descriptor_source_transform.a * descriptor_source_transform.e
    - descriptor_source_transform.b * descriptor_source_transform.d
)

# A cell enters the summary only when both products are mapped
combined_valid_mask = (
    (desc_array != ANNUAL_NLCD_NODATA)
    & (imp_array != ANNUAL_NLCD_NODATA)
)
desc_values, desc_counts = np.unique(
    desc_array[combined_valid_mask],
    return_counts=True,
)

# Translate the three Annual NLCD descriptor codes into clear labels
descriptor_names = {
    0: 'Non-urban / not impervious',
    1: 'Roads',
    2: 'Urban (other built surfaces)',
}

descriptor_rows = []
total_descriptor_cells = desc_counts.sum()

# Summarize both land coverage and equivalent impervious contribution
for code_value, count in zip(desc_values, desc_counts):
    class_mask = combined_valid_mask & (desc_array == code_value)

    # Weight each cell by its fractional impervious percentage
    class_impervious_area_m2 = (
        (imp_array[class_mask].astype(float) / 100)
        * descriptor_cell_area_m2
    ).sum()

    descriptor_rows.append({
        'descriptor_code': int(code_value),
        'descriptor_class': descriptor_names.get(
            int(code_value), f'code {code_value}'
        ),
        'pixel_count': int(count),
        'percent_mapped_cells': 100 * count / total_descriptor_cells,
        'impervious_area_acres': class_impervious_area_m2 / 4046.86,
    })

# Convert accumulated records into a tidy engineering table
descriptor_summary = pd.DataFrame(descriptor_rows)
descriptor_total_impervious_acres = (
    descriptor_summary['impervious_area_acres'].sum()
)
descriptor_summary['percent_total_impervious_area'] = (
    100
    * descriptor_summary['impervious_area_acres']
    / descriptor_total_impervious_acres
)
descriptor_summary = descriptor_summary.sort_values(
    'impervious_area_acres', ascending=False
).reset_index(drop=True)

# Reconcile the descriptor-weighted sum with the Part 6 watershed total
descriptor_area_difference_pct = (
    100
    * abs(
        descriptor_total_impervious_acres
        - total_impervious_area_acres
    )
    / total_impervious_area_acres
)

print(descriptor_summary[[
    'descriptor_class',
    'percent_mapped_cells',
    'impervious_area_acres',
    'percent_total_impervious_area',
]].round(2).to_string(index=False))
print(
    f"\nDescriptor-weighted impervious area check: "
    f"{descriptor_total_impervious_acres:,.1f} acres"
)
print(f"Difference from Part 6 total: {descriptor_area_difference_pct:.2f}%")

if descriptor_area_difference_pct > 1:
    print(
        "WARNING: Descriptor and fractional impervious totals differ "
        "by more than 1%. Investigate before interpreting the split."
    )
else:
    print("Descriptor-weighted area QA/QC passed.")


### What the Descriptor Summary Shows

`percent_mapped_cells` describes land coverage. `percent_total_impervious_area` answers the engineering question: how much equivalent impervious area is associated with roads or other built surfaces?

The descriptor-weighted total should reconcile with the Part 6 watershed total. A difference above 1 percent triggers a warning because it may indicate grid, mask, or NoData problems.

**Question to ask your AI assistant:** *"Explain why descriptor classes should be weighted by fractional impervious area instead of summarized only by pixel count. Use one low-impervious and one high-impervious cell in your example."*


## Part 8: Annual Change Analysis - 2001 versus 2025

### Compare Land Cover Classes

The baseline raster is clipped with the same watershed geometry and background rule. Before calculating change, the code checks that baseline and current arrays have the same shape and transform.

An outer join keeps classes that occur in only one year. Change is reported in percentage points, which is current watershed share minus baseline watershed share.


In [ ]:
# Clip the baseline land cover to the same watershed geometry
lc_baseline_path = f'annual_nlcd_{BASELINE_YEAR}_land_cover.tif'

with rasterio.open(lc_baseline_path) as src:
    lc_baseline_clipped, lc_baseline_transform = rasterio.mask.mask(
        src,
        watershed_geom,
        crop=True,
        nodata=ANNUAL_NLCD_NODATA,
    )

# Confirm both years use identical cells before calculating change
lc_baseline_array = lc_baseline_clipped[0]
if (
    lc_baseline_array.shape != lc_array.shape
    or not lc_baseline_transform.almost_equals(lc_transform)
):
    raise RuntimeError(
        "Baseline and current land cover rasters are not on the same grid."
    )

# Count baseline land cover classes with the same NoData rule
class_values_baseline, pixel_counts_baseline = np.unique(
    lc_baseline_array[
        lc_baseline_array != ANNUAL_NLCD_NODATA
    ],
    return_counts=True,
)

# Build the baseline percentage table before joining the two years
lc_baseline_summary = pd.DataFrame({
    'nlcd_code': class_values_baseline,
    'nlcd_class': [
        nlcd_classes.get(int(code), f'Unknown ({code})')
        for code in class_values_baseline
    ],
    'pixel_count': pixel_counts_baseline,
    'percent_watershed_baseline': (
        100
        * pixel_counts_baseline
        / pixel_counts_baseline.sum()
    ),
})

# An outer merge keeps classes that occur in only one year
comparison = lc_summary[[
    'nlcd_code', 'percent_watershed'
]].merge(
    lc_baseline_summary[[
        'nlcd_code', 'percent_watershed_baseline'
    ]],
    on='nlcd_code',
    how='outer',
).rename(columns={
    'percent_watershed': 'percent_watershed_current'
})

# Missing percentages mean the class occupied zero mapped area that year
comparison[[
    'percent_watershed_baseline',
    'percent_watershed_current',
]] = comparison[[
    'percent_watershed_baseline',
    'percent_watershed_current',
]].fillna(0)

# Restore class names after the outer merge
comparison['nlcd_class'] = comparison['nlcd_code'].map(
    lambda code: nlcd_classes.get(
        int(code), f'Unknown ({code})'
    )
)

# Percentage-point change is current share minus baseline share
comparison['change_pct_points'] = (
    comparison['percent_watershed_current']
    - comparison['percent_watershed_baseline']
)
comparison = comparison[[
    'nlcd_code',
    'nlcd_class',
    'percent_watershed_baseline',
    'percent_watershed_current',
    'change_pct_points',
]].sort_values(
    'percent_watershed_current', ascending=False
)

print(comparison.round(2).to_string(index=False))



The comparison table is sorted by current watershed share. Positive values gained percentage points and negative values lost percentage points.

Do not interpret the changes yet. The next code cell checks whether both years have nearly the same valid mapped area. That check protects the comparison denominator.


In [ ]:
# QA/QC: confirm both years represent nearly the same mapped area
# Build an independent mapped-area record for each land cover year
land_cover_valid_area_check = pd.DataFrame({
    'nlcd_year': [BASELINE_YEAR, CURRENT_YEAR],
    'valid_cell_count': [
        pixel_counts_baseline.sum(),
        total_pixels,
    ],
})
land_cover_valid_area_check['valid_area_acres'] = (
    land_cover_valid_area_check['valid_cell_count']
    * cell_area_acres
)
land_cover_valid_area_check['percent_of_polygon_area'] = (
    100
    * land_cover_valid_area_check['valid_area_acres']
    / area_acres
)

# Compare coverage between years before interpreting class changes
land_cover_valid_area_change_pct = 100 * (
    land_cover_valid_area_check.loc[1, 'valid_area_acres']
    - land_cover_valid_area_check.loc[0, 'valid_area_acres']
) / land_cover_valid_area_check.loc[0, 'valid_area_acres']

print("Land cover valid-area QA/QC:")
print(land_cover_valid_area_check.round(2).to_string(index=False))
if abs(land_cover_valid_area_change_pct) > 1:
    print(
        "WARNING: Valid mapped area differs by more than 1% "
        "between years. Investigate before interpreting change."
    )
else:
    print(
        "Valid-area QA/QC passed: mapped area differs by "
        "no more than 1% between years."
    )


### Check Land Cover Coverage, Then Compare Imperviousness

The land cover valid-area table should show nearly identical mapped area in both years. If coverage differs materially, some apparent change may be a data-coverage artifact.

The next cell repeats the corrected fractional impervious calculation for the baseline year, checks grid alignment, compares valid area, and reports the change in watershed-average imperviousness.


In [ ]:
# Clip baseline fractional imperviousness with the same background value
imp_baseline_path = f'annual_nlcd_{BASELINE_YEAR}_impervious.tif'

with rasterio.open(imp_baseline_path) as src:
    imp_baseline_clipped, imp_baseline_transform = rasterio.mask.mask(
        src,
        watershed_geom,
        crop=True,
        nodata=ANNUAL_NLCD_NODATA,
    )

# Require the baseline and current rasters to share one grid
imp_baseline_array = imp_baseline_clipped[0]
if (
    imp_baseline_array.shape != imp_array.shape
    or not imp_baseline_transform.almost_equals(imp_transform)
):
    raise RuntimeError(
        "Baseline and current impervious rasters are not on the same grid."
    )

# Keep 0 through 100 and exclude only Annual NLCD background 250
imp_baseline_valid = imp_baseline_array[
    imp_baseline_array != ANNUAL_NLCD_NODATA
].astype(float)

# Calculate baseline equivalent impervious and valid mapped areas
impervious_area_baseline_m2 = (
    (imp_baseline_valid / 100) * impervious_cell_area_m2
).sum()
valid_area_baseline_m2 = (
    imp_baseline_valid.size * impervious_cell_area_m2
)
percent_impervious_baseline = (
    100 * impervious_area_baseline_m2 / valid_area_baseline_m2
)

# Compare valid coverage before interpreting temporal change
impervious_valid_area_check = pd.DataFrame({
    'nlcd_year': [BASELINE_YEAR, CURRENT_YEAR],
    'valid_cell_count': [
        imp_baseline_valid.size,
        imp_valid.size,
    ],
    'valid_area_acres': [
        valid_area_baseline_m2 / 4046.86,
        total_valid_area_acres,
    ],
})
impervious_valid_area_change_pct = 100 * (
    impervious_valid_area_check.loc[1, 'valid_area_acres']
    - impervious_valid_area_check.loc[0, 'valid_area_acres']
) / impervious_valid_area_check.loc[0, 'valid_area_acres']

print("Impervious valid-area QA/QC:")
print(impervious_valid_area_check.round(2).to_string(index=False))
if abs(impervious_valid_area_change_pct) > 1:
    print(
        "WARNING: Valid mapped area differs by more than 1% "
        "between years. Investigate before interpreting change."
    )
else:
    print(
        "Valid-area QA/QC passed: mapped area differs by "
        "no more than 1% between years."
    )

# Report change as percentage points over the actual year interval
elapsed_years = CURRENT_YEAR - BASELINE_YEAR
print(
    f"{BASELINE_YEAR} watershed-average percent impervious: "
    f"{percent_impervious_baseline:.2f}%"
)
print(
    f"{CURRENT_YEAR} watershed-average percent impervious: "
    f"{percent_impervious_current:.2f}%"
)
print(
    f"Change over {elapsed_years} years: "
    f"{percent_impervious_current - percent_impervious_baseline:+.2f} "
    "percentage points"
)


### Review the Two Land Cover Maps

The side-by-side maps use the same official class colors and hide background cells. Compare the location and extent of developed classes rather than relying only on the watershed-wide percentages.

The maps do not prove that every changed cell is correct. Use them to identify patterns worth checking against imagery or local development records.


In [ ]:
# Create one map axis for each comparison year
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

map_arrays = [lc_baseline_array, lc_array]
map_years = [BASELINE_YEAR, CURRENT_YEAR]

# Apply the same class colors independently to each year's array
for axis, array, year in zip(axes, map_arrays, map_years):
    codes_present = sorted(np.unique(
        array[array != ANNUAL_NLCD_NODATA]
    ).tolist())
    colors_present = [
        nlcd_colors[code] for code in codes_present
    ]
    cmap_year = ListedColormap(colors_present)
    index_lookup = {
        code: index for index, code in enumerate(codes_present)
    }
    # Convert class codes to consecutive positions used by Matplotlib
    display_array = np.vectorize(
        lambda value: index_lookup.get(value, -1)
    )(array)
    display_array = np.ma.masked_where(
        array == ANNUAL_NLCD_NODATA, display_array
    )

    axis.imshow(
        display_array,
        cmap=cmap_year,
        vmin=0,
        vmax=len(codes_present) - 1,
    )
    axis.set_title(f"Annual NLCD {year} Land Cover")
    axis.set_xticks([])
    axis.set_yticks([])

# The shared title states the watershed and comparison interval
fig.suptitle(
    f"{target_watershed['Name'].iloc[0]}: "
    f"{BASELINE_YEAR} vs. {CURRENT_YEAR}",
    fontsize=14,
)
plt.tight_layout()
plt.show()


### What the Change Analysis Shows

The tables quantify changes in class share and fractional imperviousness over 24 years. The maps show where the land cover pattern changed.

Because both years come from Annual NLCD Collection 1.2 and the grid and valid-area checks pass, the comparison has a consistent basis. It remains a screening result. Review unexpected transitions against imagery, local GIS, and known development history.

**Question to ask your AI assistant:** *"Using the comparison table and impervious summary, distinguish percentage change from percentage-point change. Which one is printed here, and why is that wording important in an engineering report?"*


## Part 9: Export Traceable Tables and Rasters

The next cell creates three CSV files: land cover by class and year, watershed imperviousness by year, and the current descriptor summary.

Each export records source, collection version, DOI, year, access date, and processing notes. These fields prevent Annual NLCD results from being confused with legacy NLCD or another collection version later.


In [ ]:
# Record source identity once and attach it to every exported table
access_date = datetime.date.today().isoformat()
collection_version = 'Annual NLCD Collection 1.2'
source_doi = 'https://doi.org/10.5066/P94UXNTS'

# Land cover summary for both years
# Prepare current and baseline land cover rows in one common schema
lc_current_export = lc_summary.copy()
lc_current_export['nlcd_year'] = CURRENT_YEAR

lc_baseline_export = lc_baseline_summary.rename(columns={
    'percent_watershed_baseline': 'percent_watershed',
}).copy()
lc_baseline_export['area_acres'] = (
    lc_baseline_export['pixel_count'] * cell_area_acres
)
lc_baseline_export['nlcd_year'] = BASELINE_YEAR

# Stack both years and add fields needed for traceability
lc_export_combined = pd.concat(
    [lc_baseline_export, lc_current_export],
    ignore_index=True,
)
lc_export_combined['watershed_id'] = target_huc
lc_export_combined['source'] = 'USGS Annual NLCD Land Cover'
lc_export_combined['collection_version'] = collection_version
lc_export_combined['source_doi'] = source_doi
lc_export_combined['access_date'] = access_date
lc_export_combined['processing_notes'] = (
    'Clipped to watershed; NoData=250 excluded; '
    'cell area read from affine transform'
)

lc_export_combined.to_csv(
    'watershed_land_cover_summary.csv', index=False
)
print(
    f"Saved watershed_land_cover_summary.csv "
    f"({len(lc_export_combined)} rows)"
)

# Fractional impervious summary for both years
# Keep one watershed-level impervious record per analysis year
impervious_summary = pd.DataFrame([
    {
        'watershed_id': target_huc,
        'nlcd_year': BASELINE_YEAR,
        'mean_impervious_percent': round(
            percent_impervious_baseline, 2
        ),
        'equivalent_impervious_area_acres': round(
            impervious_area_baseline_m2 / 4046.86, 1
        ),
        'total_valid_area_acres': round(
            valid_area_baseline_m2 / 4046.86, 1
        ),
    },
    {
        'watershed_id': target_huc,
        'nlcd_year': CURRENT_YEAR,
        'mean_impervious_percent': round(
            percent_impervious_current, 2
        ),
        'equivalent_impervious_area_acres': round(
            total_impervious_area_acres, 1
        ),
        'total_valid_area_acres': round(
            total_valid_area_acres, 1
        ),
    },
])
impervious_summary['source'] = (
    'USGS Annual NLCD Fractional Impervious Surface'
)
impervious_summary['collection_version'] = collection_version
impervious_summary['source_doi'] = source_doi
impervious_summary['processing_notes'] = (
    'NoData=250 excluded; 0 is valid; impervious-area weighted'
)
impervious_summary['access_date'] = access_date

impervious_summary.to_csv(
    'watershed_impervious_summary.csv', index=False
)
print(
    f"Saved watershed_impervious_summary.csv "
    f"({len(impervious_summary)} rows)"
)

# Current-year descriptor summary
# Export the current descriptor table with the same source metadata
descriptor_summary_export = descriptor_summary.copy()
descriptor_summary_export['watershed_id'] = target_huc
descriptor_summary_export['nlcd_year'] = CURRENT_YEAR
descriptor_summary_export['source'] = (
    'USGS Annual NLCD Impervious Descriptor'
)
descriptor_summary_export['collection_version'] = collection_version
descriptor_summary_export['source_doi'] = source_doi
descriptor_summary_export['processing_notes'] = (
    'NoData=250 excluded; classes weighted by fractional impervious area'
)
descriptor_summary_export['access_date'] = access_date
descriptor_summary_export.to_csv(
    'watershed_impervious_descriptor_summary.csv',
    index=False,
)
print(
    "Saved watershed_impervious_descriptor_summary.csv "
    f"({len(descriptor_summary_export)} rows)"
)

impervious_summary


The CSV status messages should report 20 land cover rows, 2 impervious rows, and 3 descriptor rows for this watershed. Open the tables after download and confirm the year and source fields are populated.

### Export the Current Clipped Rasters

The next cell writes the exact current-year arrays used in the calculations. Each GeoTIFF keeps its own clip transform, CRS, NoData value, and compression. The land cover export also keeps its color table.


In [ ]:
# Build output names that retain the product year
clipped_land_cover_filename = (
    f'clipped_annual_nlcd_land_cover_{CURRENT_YEAR}.tif'
)
clipped_impervious_filename = (
    f'clipped_annual_nlcd_impervious_{CURRENT_YEAR}.tif'
)

# Export current land cover with its source color table
with rasterio.open(lc_current_path) as src:
    output_profile = src.profile.copy()
    output_profile.update(
        height=lc_array.shape[0],
        width=lc_array.shape[1],
        transform=lc_transform,
        nodata=ANNUAL_NLCD_NODATA,
        compress='lzw',
    )
    try:
        source_colormap = src.colormap(1)
    except ValueError:
        source_colormap = None

with rasterio.open(
    clipped_land_cover_filename, 'w', **output_profile
) as dst:
    dst.write(lc_array, 1)
    if source_colormap is not None:
        dst.write_colormap(1, source_colormap)

# Export current fractional impervious surface with its own transform
with rasterio.open(imp_current_path) as src:
    output_profile = src.profile.copy()
    output_profile.update(
        height=imp_array.shape[0],
        width=imp_array.shape[1],
        transform=imp_transform,
        nodata=ANNUAL_NLCD_NODATA,
        compress='lzw',
    )

with rasterio.open(
    clipped_impervious_filename, 'w', **output_profile
) as dst:
    dst.write(imp_array, 1)

print(
    f"Saved {clipped_land_cover_filename} and "
    f"{clipped_impervious_filename}"
)



The saved rasters are analysis outputs, not new source products. Their filenames include the year, and their metadata retain the Annual NLCD grid and background value.

### Download the Lesson Outputs

The next cell starts one Colab download for each CSV and GeoTIFF. Keep the files together so the tables, rasters, source version, and processing notes remain traceable.


In [ ]:
# Download all lesson outputs to your computer
from google.colab import files

output_files = [
    'watershed_land_cover_summary.csv',
    'watershed_impervious_summary.csv',
    'watershed_impervious_descriptor_summary.csv',
    clipped_land_cover_filename,
    clipped_impervious_filename,
]

for output_file in output_files:
    files.download(output_file)


### Connecting Forward

This module's land cover table and Module 7's hydrologic soil group table are the two inputs a full composite curve number workflow needs. The fractional impervious result can also support HEC-HMS parameter screening, but it is not a calibrated directly connected impervious area. Confirm connectivity, drainage, and local development data before design use.


## Part 10: Beyond Annual NLCD - Coastal and Local Alternatives

### NOAA C-CAP

NOAA C-CAP provides regional and high-resolution land cover products focused on coastal areas. It uses a different classification scheme and coverage. Evaluate it for coastal watersheds where those products better match the project scale.

### Local and Project-Specific Data

Annual NLCD's 30 meter resolution is appropriate for watershed and regional screening. It is too coarse for parcel-scale design. Local municipal impervious layers, building footprints, roadway inventories, and recent imagery may be better for detailed drainage studies.

### Source Hierarchy Summary

| If you need... | Use |
|---|---|
| A consistent annual CONUS land cover and impervious series | Annual NLCD |
| Coastal-specific or higher-resolution coastal land cover | NOAA C-CAP |
| Parcel-scale design detail | Local GIS, imagery, and as-built data |


## Engineering Cautions

1. **Categorical rasters hold codes, not quantities.** Never average land cover or descriptor class codes.
2. **Use fractional imperviousness for percent impervious.** Developed land cover classes include permeable surfaces.
3. **Annual NLCD background is 250 for these products.** Values from 0 to 100 are valid fractional impervious percentages. Descriptor values 0, 1, and 2 are also valid.
4. **Keep grids aligned before cell-by-cell work.** Check CRS, shape, transform, resolution, and valid area.
5. **Reproject vectors to match the raster.** Resampling a categorical raster can change class assignments at boundaries.
6. **Thirty-meter data support screening, not parcel-scale design.**
7. **Annual NLCD is not directly connected impervious area.** Drainage connectivity needs separate engineering judgment.
8. **Document collection version and year.** Annual updates can extend and revise the time series.


## Troubleshooting

| Problem | Likely Cause | What to Try |
|---|---|---|
| Year discovery fails | Temporary service or network issue | The notebook uses the Collection 1.2 year list and continues |
| Repeated `500`/`503` errors or timeouts | USGS server outage or maintenance, not your code | Let the run continue to the GitHub fallback, or rerun the cell later |
| WCS returns XML instead of a raster | Invalid request or service issue | Read the printed error, then rerun the fetch cell |
| A fetch is slow | Larger areas need more tiles, and each tile is a separate request | Expect longer waits for big watersheds; confirm one HUC-12 is selected and reduce `buffer_m` if appropriate |
| Local fallback is rejected | It does not cover the requested watershed or failed validation | Let the helper continue to the live WCS |
| Clipped raster is empty | Watershed is outside CONUS coverage or CRS is wrong | Print watershed and raster CRS and bounds |
| Land cover and impervious grids do not align | Different bounds, transforms, or products were mixed | Rerun all five requests with the same `request_bounds` |
| Percent impervious is too high | Legitimate 0 values were excluded | Use 250 as Annual NLCD background |
| Descriptor totals do not match | Grids, masks, or NoData handling differ | Check alignment and combined valid mask before interpreting |
| Colab upload fails | The watershed ZIP was not selected | Rerun the upload cell and select the course file |


## Practice Exercises

Each exercise can be completed by lightly editing code already used in the lesson.

### Exercise 1: A Different Watershed

Change `target_huc` to another HUC-12 in the course file. Rerun the watershed selection, request bounds, fetch, clip, and summary cells. The local fallback covers the course watersheds, and the live WCS supports other CONUS watersheds when a suitable local file is not present.


In [ ]:
# EXERCISE 1: A different watershed
# Your code here.
#
# Steps:
# 1. Choose another HUC-12 code from the watersheds table.
# 2. Re-select target_watershed and rebuild request_bounds.
# 3. Call fetch_annual_nlcd_raster() for the five products.
# 4. Reproject the watershed to the raster CRS.
# 5. Rerun the clip, area, and valid-coverage checks.


### Exercise 2: Impervious Hotspots

Using `imp_array`, count valid cells above 50 percent impervious and convert their full cell footprint to acres. Explain why this hotspot footprint is different from equivalent impervious area.


In [ ]:
# EXERCISE 2: Impervious hotspots
# Your code here.
#
# Steps:
# 1. Build a mask for values greater than 50 and not equal to 250.
# 2. Count cells meeting both conditions.
# 3. Multiply by cell_area_acres for hotspot footprint area.
# 4. Explain why this is not equivalent impervious area.


### Exercise 3: Road Share of Impervious Area

Use `descriptor_summary` to report the percentage of equivalent impervious area assigned to roads, code 1, versus urban or other built surfaces, code 2. Keep code 0 visible as a QA/QC category.


In [ ]:
# EXERCISE 3: Descriptor road percentage
# Your code here.
#
# Steps:
# 1. Select the row where descriptor_code equals 1 for roads.
# 2. Select descriptor_code 2 for other built surfaces.
# 3. Report percent_total_impervious_area for both rows.
# 4. Check whether code 0 contributes meaningful impervious area.


### Challenge Exercise: Choose Another Annual Year

Change `CURRENT_YEAR` to another available year and rerun the fetch and analysis workflow. Use a filename that includes the new year. Compare the new result with 2025 and explain why collection version, valid area, and grid alignment still need to be documented.

Suggested prompt:

*"Help me adapt my Annual NLCD notebook so I can compare any two years returned by available_annual_nlcd_years(). Keep land cover categorical, keep fractional impervious continuous, use NoData 250, and require grid and valid-area checks before calculating change."*


In [ ]:
# CHALLENGE EXERCISE: choose another Annual NLCD year
# Your code here.
#
# Steps:
# 1. Print available_years and choose a different year.
# 2. Build year-specific filenames for land cover and imperviousness.
# 3. Fetch both products with fetch_annual_nlcd_raster().
# 4. Clip them to the watershed and verify grid alignment.
# 5. Compare class percentages and mean imperviousness with 2025.


## That's Module 8 Done!

You used free, anonymous USGS web services to discover and retrieve Annual NLCD data without an AWS account. You validated the raster responses, kept categorical and continuous products on separate calculation paths, caught a NoData mistake, summarized impervious descriptors by impervious-area weighting, and compared two years from one consistent annual product series.

### What you can do now

- Fetch Annual NLCD land cover and impervious products for a CONUS watershed
- Verify CRS, resolution, transform, NoData, value range, and valid area
- Summarize categorical land cover by area
- Calculate watershed-average fractional imperviousness and equivalent impervious area
- Separate road and other built contributions using impervious-area weighting
- Compare any two Annual NLCD years with clear QA/QC

### Next Steps

- Combine Annual NLCD land cover with Module 7 hydrologic soil groups for curve number workflows.
- Compare the screening results with local impervious and development data.
- Use local or higher-resolution sources when project scale requires more detail.

### Official Resources

- [Annual NLCD Product Suite](https://www.usgs.gov/centers/eros/science/nlcd-product-suite)
- [Annual NLCD Collection 1.2 Data Release](https://www.usgs.gov/data/annual-national-land-cover-database-nlcd-collection-1-products)
- [Annual NLCD Collection 1.2 User Guide](https://www.mrlc.gov/sites/default/files/docs/LSDS-2103%20Annual%20National%20Land%20Cover%20Database%20%28NLCD%29%20Collection%201%20Science%20Product%20User%20Guide%20-v1.2%202026_04_21.pdf)
- [MRLC Data Services](https://www.mrlc.gov/data-services-page)
- [Fractional Impervious Surface](https://www.mrlc.gov/data/type/fractional-impervious-surface)
- [Impervious Descriptor](https://www.mrlc.gov/data/type/impervious-descriptor)
